# Generador de simulaciones Quantum ESPRESSO

Crea una carpeta por temperatura y, dentro, una subcarpeta por cada combinación de
`dt`, `mixing_beta` y `degauss`. En cada una deja `input.in`, `job_BSC.sh` y los pseudopotenciales.
Los casos repetidos se numeran como `_sim1`, `_sim2`... y los nombres no llevan puntos (`0.15` → `15`).
También crea `submit_all.sh` para lanzar todo con `sbatch`.

**Antes de empezar:** el notebook debe estar en la misma carpeta que `input.in`, `job_BSC.sh`
y los `.UPF` (o cambia `SRC` en la primera celda).

Orden de uso: **1)** ejecutar celdas 1 y 2 → **2)** vista previa → **3)** crear → **4)** (opcional) comprobar cambios.

## 1. Configuración (lo único que hay que editar)

In [ ]:
from pathlib import Path
import re
import shutil
from collections import Counter

# ---------------- Rutas ----------------
SRC = Path(".")               # carpeta con input.in, job_BSC.sh y los .UPF
OUT = Path("Dataset_BGDY3x3_AIMD_v3")    # carpeta que se va a crear
INPUT_NAME = "input.in"
JOB_NAME = "job_BSC.sh"

# ---------------- Parámetros ----------------
# Prefijo de QE; {T} se sustituye por la temperatura
PREFIX_TEMPLATE = "BGDY_3x3_{T}"

# Valores por defecto (los del input.in original)
DEFAULTS = {
    "dt": 40,              # se escribe como 40.d0
    "mixing_beta": 0.2,
    "degauss": 0.005,      # Ry
}

# Variantes de convergencia reutilizables
V_BASE   = {"mixing_beta": 0.2,  "degauss": 0.005}   # por defecto
V_BETA   = {"mixing_beta": 0.15, "degauss": 0.005}
V_BETA_G = {"mixing_beta": 0.15, "degauss": 0.01}

# Casos por temperatura (K). Cada elemento de la lista = una subcarpeta.
# Si un mismo caso se repite, las carpetas se numeran solas: ..._sim1, ..._sim2, ...
CASES = {
    1750: [
        {**V_BASE},
        {**V_BASE},
        {**V_BASE},
        {**V_BASE},
        {**V_BASE},
    ],
    1500: [
        {**V_BASE},
        {**V_BASE},
        {**V_BASE},
        {**V_BASE},
        {**V_BASE},
    ],
    1000: [
        {**V_BASE},
        {**V_BASE},
        {**V_BASE},
        {**V_BASE},
        {**V_BASE},
    ],
    500: [
        {**V_BASE},
        {**V_BASE},
        {**V_BASE},
        {**V_BASE},
        {**V_BASE},
    ],
}

## 2. Funciones (no hace falta tocar nada)

In [ ]:
def fmt_dt(dt):
    """Formato Fortran de doble precisión: 40 -> '40.d0', 12.5 -> '12.5d0'."""
    dt = float(dt)
    return f"{int(dt)}.d0" if dt.is_integer() else f"{dt:g}d0"


def set_param(texto, clave, valor):
    """Cambia `clave = valor` respetando el resto de la línea. Falla si la clave
    no aparece exactamente una vez (así nunca hay cambios silenciosos)."""
    patron = re.compile(rf"^([ \t]*{re.escape(clave)}[ \t]*=[ \t]*)([^,\s]+)", re.MULTILINE)
    nuevo, n = patron.subn(lambda m: m.group(1) + valor, texto)
    if n != 1:
        raise ValueError(f"'{clave}' aparece {n} veces en el input (se esperaba 1).")
    return nuevo


def completar(caso):
    """Mezcla los valores por defecto con las modificaciones del caso."""
    p = dict(DEFAULTS)
    p.update(caso)
    return p


def num(x):
    """Número sin punto, quedándose con lo que va detrás:
    0.15 -> '15', 0.005 -> '005', 0.2 -> '2', 20 -> '20'."""
    s = f"{x:g}"
    if "e" in s:                                  # evita 1e-05
        s = f"{x:.12f}".rstrip("0")
    if s.startswith("0."):
        return s[2:]
    return s.replace(".", "p")                    # caso raro, p.ej. 1.5 -> '1p5'


def etiqueta(p):
    """Nombre base de la subcarpeta, p.ej. dt20_beta15_degauss01"""
    return f"dt{num(p['dt'])}_beta{num(p['mixing_beta'])}_degauss{num(p['degauss'])}"


def plan_carpetas(T):
    """Lista de (parámetros, nombre_de_carpeta) para la temperatura T.
    Los casos con parámetros idénticos se numeran: _sim1, _sim2, ..."""
    params = [completar(c) for c in CASES[T]]
    tags = [etiqueta(p) for p in params]

    # parámetros distintos no pueden dar el mismo nombre
    vistos = {}
    for p, tag in zip(params, tags):
        clave = tuple(sorted(p.items()))
        if vistos.setdefault(tag, clave) != clave:
            raise ValueError(f"En T={T} K, parámetros distintos dan el mismo nombre ({tag}).")

    total, contador, plan = Counter(tags), Counter(), []
    for p, tag in zip(params, tags):
        if total[tag] > 1:
            contador[tag] += 1
            plan.append((p, f"{tag}_sim{contador[tag]}"))
        else:
            plan.append((p, tag))
    return plan


def construir_input(plantilla, T, p):
    txt = plantilla
    txt = set_param(txt, "prefix", f"'{PREFIX_TEMPLATE.format(T=T)}'")
    txt = set_param(txt, "dt", fmt_dt(p["dt"]))
    txt = set_param(txt, "degauss", f"{p['degauss']:g}")
    txt = set_param(txt, "mixing_beta", f"{p['mixing_beta']:g}")
    txt = set_param(txt, "tempw", f"{float(T):.1f}")
    return txt


def construir_job(plantilla, T, nombre_carpeta):
    nombre = f"qe{T}_{nombre_carpeta}"
    nuevo, n = re.subn(r"^#SBATCH\s+--job-name=.*$", f"#SBATCH --job-name={nombre}",
                       plantilla, count=1, flags=re.MULTILINE)
    if n != 1:
        raise ValueError("No encuentro la línea '#SBATCH --job-name=' en el job script.")
    return nuevo


def pseudos_del_input(texto):
    """Nombres de los pseudopotenciales en la tarjeta ATOMIC_SPECIES."""
    nombres, dentro = [], False
    for linea in texto.splitlines():
        s = linea.strip()
        if s.upper().startswith("ATOMIC_SPECIES"):
            dentro = True
            continue
        if dentro:
            if not s:
                continue
            campos = s.split()
            if len(campos) >= 3 and re.match(r"^[\d.]+([dDeE][+-]?\d+)?$", campos[1]):
                nombres.append(campos[2])
            else:
                break
    if not nombres:
        raise ValueError("No he encontrado ATOMIC_SPECIES en el input.")
    return nombres


def _norm(nombre):
    return re.sub(r"[._]", "", nombre.lower())


def buscar_pseudo(src, nombre):
    """Busca el pseudo en `src`; acepta '.' o '_' indistintamente
    (C.pbe-n-kjpaw_psl.1.0.0.UPF ~ C_pbe-n-kjpaw_psl_1_0_0.UPF)."""
    exacto = src / nombre
    if exacto.is_file():
        return exacto
    for f in src.iterdir():
        if f.is_file() and _norm(f.name) == _norm(nombre):
            return f
    raise FileNotFoundError(
        f"No encuentro el pseudopotencial '{nombre}' en {src}. "
        f"Archivos .UPF disponibles: {[f.name for f in src.glob('*.UPF')]}"
    )


def escribir(ruta, texto):
    """Escribe con saltos de línea Unix (\n) aunque el notebook corra en Windows.
    Con \r\n, bash falla en el clúster ('/bin/bash^M: bad interpreter')."""
    with open(ruta, "w", encoding="utf-8", newline="\n") as f:
        f.write(texto.replace("\r\n", "\n"))


def revisar_saltos_de_linea(out):
    """Avisa si algún .sh o .in generado contiene saltos de línea de Windows."""
    malos = [f for f in Path(out).rglob("*")
             if f.is_file() and f.suffix in (".sh", ".in") and b"\r" in f.read_bytes()]
    if malos:
        print("AVISO: saltos de línea de Windows en:", *malos, sep="\n  ")
    else:
        print("Saltos de línea OK (formato Unix) en todos los .sh y .in.")


SUBMIT_ALL = """#!/bin/bash
# Lanza con sbatch todos los casos (o solo los de una temperatura).
#   ./submit_all.sh          -> todos
#   ./submit_all.sh 2750     -> solo T2750K
cd "$(dirname "$0")" || exit 1
for d in T${1:-*}K/*/; do
    if [ -f "$d/job_BSC.sh" ]; then
        echo "Enviando $d"
        ( cd "$d" && sbatch job_BSC.sh )
    fi
done
"""


def crear_simulaciones(src=SRC, out=OUT, dry_run=False, force=False):
    """Crea todas las carpetas. dry_run=True solo muestra lo que haría;
    force=True sobrescribe input.in y job_BSC.sh en subcarpetas ya existentes."""
    src, out = Path(src).resolve(), Path(out).resolve()

    for nombre in (INPUT_NAME, JOB_NAME):
        if not (src / nombre).is_file():
            raise FileNotFoundError(f"No encuentro {src / nombre}")

    plantilla_input = (src / INPUT_NAME).read_text(encoding="utf-8")
    plantilla_job = (src / JOB_NAME).read_text(encoding="utf-8")
    pseudos = {n: buscar_pseudo(src, n) for n in pseudos_del_input(plantilla_input)}

    print(f"Origen : {src}")
    print(f"Destino: {out}")
    print("Pseudos: " + ", ".join(f"{k} <- {v.name}" for k, v in pseudos.items()))
    print()

    total = 0
    for T in CASES:
        for p, nombre in plan_carpetas(T):
            carpeta = out / f"T{T}K" / nombre
            existe = carpeta.exists()
            estado = "sobrescribe" if (existe and force) else ("YA EXISTE, se omite" if existe else "nuevo")
            print(f"  T{T}K/{nombre:<34} [{estado}]")
            total += 1

            if dry_run or (existe and not force):
                continue

            txt_input = construir_input(plantilla_input, T, p)
            txt_job = construir_job(plantilla_job, T, nombre)

            carpeta.mkdir(parents=True, exist_ok=True)
            escribir(carpeta / "input.in", txt_input)
            escribir(carpeta / "job_BSC.sh", txt_job)
            shutil.copymode(src / JOB_NAME, carpeta / "job_BSC.sh")
            for nombre_esperado, origen in pseudos.items():
                shutil.copyfile(origen, carpeta / nombre_esperado)

    if not dry_run:
        out.mkdir(parents=True, exist_ok=True)
        submit = out / "submit_all.sh"
        escribir(submit, SUBMIT_ALL)
        submit.chmod(0o755)

    print(f"\n{total} casos {'(simulado, no se ha escrito nada)' if dry_run else 'procesados'}.")
    if not dry_run:
        revisar_saltos_de_linea(out)

## 3. Vista previa (no escribe nada)

In [ ]:
crear_simulaciones(SRC, OUT, dry_run=True)

## 4. Crear las carpetas

Si una subcarpeta ya existe se omite (para no pisar una simulación en marcha). Usa `force=True` para sobrescribir.

In [ ]:
crear_simulaciones(SRC, OUT)

## 5. (Opcional) Comprobar qué cambia respecto a la plantilla

In [ ]:
import difflib

def ver_cambios(T, indice, src=SRC, out=OUT):
    """Diff entre la plantilla y el input.in del caso nº `indice` (desde 0) de la temperatura T."""
    _, nombre = plan_carpetas(T)[indice]
    orig = (Path(src) / INPUT_NAME).read_text().splitlines()
    nuevo = (Path(out) / f"T{T}K" / nombre / "input.in").read_text().splitlines()
    for l in difflib.unified_diff(orig, nuevo, "input.in (plantilla)", f"T{T}K/{nombre}",
                                  lineterm="", n=0):
        print(l)

ver_cambios(2750, 1)    # 2750 K, segundo caso
ver_cambios(2250, 2)    # 2250 K, tercer caso (sim3)

## 6. (Opcional) Lanzar los trabajos

Solo si el notebook se está ejecutando en el clúster (donde existe `sbatch`). Quita el `#` para usarlo.

In [ ]:
# !cd simulaciones && ./submit_all.sh          # todos
# !cd simulaciones && ./submit_all.sh 2750     # solo 2750 K